In [0]:
%sql
use catalog deltalake_catalog;

In [0]:
%fs
ls /databricks-datasets/nyctaxi/tables/nyctaxi_yellow

In [0]:
%sql
describe formatted delta.`dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow`

In [0]:
%sql
DROP TABLE IF EXISTS demo_taxi_200files;
CREATE TABLE demo_taxi_200files (
  vendor_id STRING,
  pickup_datetime TIMESTAMP,
  dropoff_datetime TIMESTAMP,
  passenger_count INT,
  trip_distance DOUBLE,
  pickup_longitude DOUBLE,
  pickup_latitude DOUBLE,
  rate_code_id INT,
  store_and_fwd_flag STRING,
  dropoff_longitude DOUBLE,
  dropoff_latitude DOUBLE,
  payment_type STRING,
  fare_amount DOUBLE,
  extra DOUBLE,
  mta_tax DOUBLE,
  tip_amount DOUBLE,
  tolls_amount DOUBLE,
  total_amount DOUBLE
)
USING DELTA
TBLPROPERTIES (
  delta.autoOptimize.optimizeWrite = false,
  delta.autoOptimize.autoCompact = false
);


In [0]:
table_location = "dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow"

# List data files (ignore _delta_log)
all_files = [f.path for f in dbutils.fs.ls(table_location) if f.path.endswith(".parquet")]

# Pick first 10 files
files10 = all_files[:10]
print("Using these 10 files:", files10)

df_10 = spark.read.parquet(*files10)

df_10.repartition(200).write.format('delta').mode("overwrite").saveAsTable("demo_taxi_200files")


Using these 10 files: ['dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-7dcc09ea-2fc1-491d-a234-3b0ce1db9336-c002.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-812df468-c3fb-405d-84d6-32cb249d8db9-c003.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-a4b4b3de-4231-4c0e-88d6-c14f202ff232-c000.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-b3d2db79-4a87-4d35-8a20-a02725fb655f-c001.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-158e21e1-9d7b-44e9-ad15-3ba7e1797de8-c002.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-9072b615-4da5-4c0c-b2c0-83f08d1944ac-c000.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-9baf2d71-1e4d-49a0-a131-03c825853637-c003.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-d61d36d1-b864-4dc2-b

In [0]:
%sql
select * from demo_taxi_200files;

In [0]:
%sql
select count(*) from demo_taxi_200files;

In [0]:
%sql
select count(*) from demo_taxi_200files where trip_distance > 100;

In [0]:
%sql
select min(trip_distance), max(trip_distance), _metadata.file_name
from demo_taxi_200files
group by _metadata.file_name
order by min(trip_distance);

min(trip_distance),max(trip_distance),file_name
0.0,930000.0,part-00030-88489887-1ffe-4a32-a875-5c185cbbc573.c000.snappy.parquet
0.0,1010015.0,part-00003-092ae74c-43d9-4069-8ab4-5eca05f2975d.c000.snappy.parquet
0.0,770.0,part-00026-595d2507-7db8-4f5a-9aec-145fbfc5d73b.c000.snappy.parquet
0.0,1000051.0,part-00048-17bd292f-2bab-4c8d-af1b-fa219411ae3f.c000.snappy.parquet
0.0,1.18000006E7,part-00036-806b0129-70cb-418f-b1e5-4929cfb87a6d.c000.snappy.parquet
0.0,184.7,part-00002-0b3efef4-62c9-4a11-af96-c2f5d103abc3.c000.snappy.parquet
0.0,50033.1,part-00045-c5aa8c20-30f7-4365-8718-821b2aaed673.c000.snappy.parquet
0.0,222.8,part-00015-a665b531-73c0-4245-80e6-2da2bf7c68e5.c000.snappy.parquet
0.0,1.0083357E7,part-00038-81ee72b9-c245-4054-9eed-898fdb7ba035.c000.snappy.parquet
0.0,756.7,part-00000-5c4b4f1e-a730-4681-874c-93c11f814d79.c000.snappy.parquet


In [0]:
%sql
optimize demo_taxi_200files
    


path,metrics
abfss://unitycatalog@ttmystorageaccount001.dfs.core.windows.net/catalog/__unitystorage/catalogs/758280e5-e8ae-48b0-840c-9f10d52cc821/tables/766eec4a-e097-47d3-b7ee-3d722f1da7a9,"List(50, 200, List(61796093, 62183384, 6.19538508E7, 50, 3097692540), List(15187155, 15273633, 1.524075144E7, 200, 3048150288), 0, null, null, 0, 1, 200, 0, true, 0, 0, 1758892346869, 1758892356482, 32, 50, null, List(0, 0), null, 18, 18, 155967, 0, null)"


In [0]:
%sql
select count(*) from demo_taxi_200files where trip_distance > 100;

count(*)
209


In [0]:
%sql
select min(trip_distance), max(trip_distance), _metadata.file_name
from demo_taxi_200files
group by _metadata.file_name
order by min(trip_distance);

In [0]:
%sql
select count(*) from demo_taxi_200files where trip_distance > 5 and trip_distance < 7;